# unit02 レッスン: BeautifulSoupで解析する

**このレッスンで作れるようになるもの**: HTML文字列を BeautifulSoup でパースして「タグのツリー」にし、`find`/`find_all` で欲しいタグを探し、`.get_text()` でテキスト・`["href"]` で属性を取り出し、`.parent`/`.find_next_sibling()` でツリーを親子・兄弟方向にたどる — スクレイピングの「解析」工程を、素の文字列処理から**パーサに任せる**やり方へ切り替えます。

unit01 では `<h1>` を `find`+スライスで切り出しました。あれは動きましたが、タグに属性が付いたり(`<h1 class="x">`)、順序が変わったりすると**すぐ壊れます**。実務のスクレイパーはほぼ例外なくパーサを使います。今日はその主役 BeautifulSoup を手に馴染ませます。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
from bs4 import BeautifulSoup


def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


# このレッスンで題材にするHTML(演習の data/blog_post.html の縮小版)。
# 本物のスクレイピングでは requests でネットから取ってきた文字列がこの変数に入る、
# と想像してください。今は「取得済みの文字列」として手元に用意しておきます。
BLOG_POST = """<!DOCTYPE html>
<html lang="ja">
<head>
    <meta charset="UTF-8">
    <title>焙煎日記 | 喫茶ポラリス</title>
</head>
<body>
    <article class="blog-post">
        <h1 class="post-title">秋の新作ブレンドができるまで</h1>
        <p class="post-meta">著者: 田中 美咲 / 投稿日: 2026-10-02</p>
        <div class="post-body">
            <p>今年は豆の乾燥に時間がかかり、焙煎の温度調整に苦労しました。</p>
            <p>深煎りと中煎りをブレンドすることで、酸味と苦味のバランスが整いました。</p>
        </div>
        <a class="post-link" href="/blog/autumn-blend">この記事の詳細ページへ</a>
    </article>
</body>
</html>"""

# ナビゲーションリンク集(演習の data/link_list.html の縮小版)
LINK_LIST = """<!DOCTYPE html>
<html lang="ja">
<head><meta charset="UTF-8"><title>ナビ | 喫茶ポラリス</title></head>
<body>
    <nav class="site-nav">
        <ul>
            <li><a href="/" class="nav-link">ホーム</a></li>
            <li><a href="/menu" class="nav-link">メニュー</a></li>
            <li><a href="/blog" class="nav-link">焙煎日記</a></li>
            <li><a href="https://external-example.test/review" class="nav-link external">口コミサイト</a></li>
        </ul>
    </nav>
</body>
</html>"""

# 入れ子になった座席案内(演習の data/nested.html の縮小版)
NESTED = """<!DOCTYPE html>
<html lang="ja">
<head><meta charset="UTF-8"><title>座席案内 | 喫茶ポラリス</title></head>
<body>
    <div class="floor" id="floor-1">
        <h2>1階フロア</h2>
        <div class="section" id="section-window">
            <span class="section-name">窓際席</span>
            <span class="seat-count">4</span>
        </div>
        <div class="section" id="section-counter">
            <span class="section-name">カウンター席</span>
            <span class="seat-count">6</span>
        </div>
    </div>
    <div class="floor" id="floor-2">
        <h2>2階フロア</h2>
        <div class="section" id="section-sofa">
            <span class="section-name">ソファ席</span>
            <span class="seat-count">8</span>
        </div>
    </div>
</body>
</html>"""

print("準備OK! BLOG_POST", len(BLOG_POST), "文字 / LINK_LIST", len(LINK_LIST), "文字 / NESTED", len(NESTED), "文字")

---
## 概念1: パースして `find` で1件探す — 文字列処理からの卒業

### なぜ学ぶか
スクレイピングの[解析]工程では「このページのタイトルは?」「著者名は?」のように**特定の1つの要素**を取り出す場面が絶えず出てきます。unit01 ではこれを `find`+スライスの手作業でやりましたが、実務では**パーサに「タグ名で探して」と頼む**のが基本です。求人票の「Webからの情報抽出」の一番地味で一番多い作業がこれです。

### 解説

**BeautifulSoup**(bs4)は、HTML文字列を解析して**タグの入れ子をオブジェクトのツリー**に変換するライブラリです。C# で XML を `XDocument.Parse(xml)` してオブジェクトツリーを得て、ノードを辿るのと同じ発想です。

```python
soup = BeautifulSoup(html文字列, "html.parser")
```

- `BeautifulSoup(...)` … HTML文字列を受け取り、ツリー全体を表す `soup` オブジェクトを返す関数。第2引数 `"html.parser"` は「Python標準のHTMLパーサを使う」という指定(いつもこれでOK)。
- `soup.find("h1")` … ツリーの中から**最初の** `<h1>` タグを1つ返す。C# の LINQ `FirstOrDefault(x => x.Name == "h1")` に相当。見つからなければ `None`。
- タグオブジェクト `tag` から:
  - `tag.get_text()` … そのタグの**中のテキスト**を返す(子タグの中身も連結される)。`strip=True` を付けると前後の空白・改行を除去。C# の `element.Value` に近い。
  - `tag.name` … タグ名の文字列(例: `"h1"`)。

unit01 では開始タグ・終了タグの位置を自力で探しましたが、**それが `soup.find("h1").get_text(strip=True)` の一撃に置き換わります**。

In [ ]:
# GOAL: HTML文字列をパースし、find で <h1> を1つ探してテキストを取り出す

# STEP 1: 文字列をパースしてツリー(soup)にする
soup = BeautifulSoup(BLOG_POST, "html.parser")
print("パース後の型:", type(soup).__name__)

# STEP 2: find で最初の <h1> タグを1つ取り出す(見つからなければ None)
h1 = soup.find("h1")
print("見つけたタグ:", repr(h1))          # タグまるごと(<h1 ...>...</h1>)が表示される
print("タグ名(.name):", h1.name)          # "h1"

# STEP 3: タグの中のテキストだけを取り出す。strip=True で前後の空白・改行を除去
print("テキスト     :", repr(h1.get_text(strip=True)))

# --- おまけ: find は「最初の1件」だけ。<p> は複数あるが最初の1個しか返らない ---
first_p = soup.find("p")
print("最初の <p>  :", repr(first_p.get_text(strip=True)))

### 予測してみよう

次のセルは `soup.find("a")`(最初の `<a>` タグ)と `soup.find("h2")`(このページに存在しない `<h2>`)を探します。

**実行する前に予測**: `<a>` タグのテキストは何になるでしょう? そして**存在しない** `<h2>` を `find` で探すと、返り値は何になるでしょう?(unit01 の `str.find` が見つからないとき `-1` を返したのと比べてみてください)

In [ ]:
# 予測してから実行!
soup = BeautifulSoup(BLOG_POST, "html.parser")

a_tag = soup.find("a")
print("最初の <a> のテキスト:", repr(a_tag.get_text(strip=True)))

h2_tag = soup.find("h2")   # このページに <h2> は無い
print("存在しない <h2> の結果:", repr(h2_tag))

存在しないタグを探すと `None` が返りました(`str.find` の `-1` と違う点に注意)。だから実務では `find` の結果を使う前に「`None` かどうか」を確認するのが定石になります(unit03 で本格的に扱います)。

### 書いてみる

**課題**: `BLOG_POST` をパースした `soup` から、`<title>` タグ(ページのタイトル)のテキストを前後空白なしで取り出し、`result1` に入れてください(期待値: `"焙煎日記 | 喫茶ポラリス"`)。

ヒント(概念レベル): `soup.find("...")` でタグを1つ取り、`.get_text(strip=True)` でテキストにする。概念1のworked exampleの `<h1>` を `<title>` に変えるだけ。

In [ ]:
soup = BeautifulSoup(BLOG_POST, "html.parser")

result1 = None
# ここに書く(result1 に代入する)


check("概念1: findでタイトル取得", result1, "焙煎日記 | 喫茶ポラリス",
      hint='soup.find("title").get_text(strip=True) の形')

---
## 概念2: `find_all` で複数まとめて集める + class で絞り込む

### なぜ学ぶか
実務でほしいデータは、たいてい**複数**あります:「ナビの全リンク」「一覧ページの全商品」「記事の全段落」。`find` は最初の1件しか返さないので、複数を扱うには `find_all` を使い、返ってきたリストを**ループで回す**のが基本パターンです。さらに「class が `external` のリンクだけ」のような**絞り込み**が、パーサの真価が出るところです。

### 解説

- `soup.find_all("a")` … 条件に合うタグを**全部リストで**返す。C# の LINQ `Where(x => x.Name == "a").ToList()` に相当。`find`(1件)に対して `find_all`(全件)。
- 返り値はリストなので、`for tag in soup.find_all("a"):` のように**ループで回して**各タグを処理します。unit01 のリスト内包表記もそのまま使えます。
- **class での絞り込み**: `soup.find_all("a", class_="external")` は「`<a>` かつ class に `external` を含む」タグだけを返します。
  - なぜ `class_`(末尾アンダースコア)かというと、`class` は Python の予約語(C# の `class` と同じくクラス定義に使う語)なので、bs4 はぶつからないように `class_` という名前にしています。**HTML の `class="..."` を絞り込みたいときは `class_=` と書く**、とだけ覚えればOK。

`len(リスト)` で件数を数えられます(C# の `.Count`)。

In [ ]:
# GOAL: find_all で <a> を全部集め、ループで回し、class で絞り込む

soup = BeautifulSoup(LINK_LIST, "html.parser")

# STEP 1: <a> タグを全部集める。返り値はリスト
all_links = soup.find_all("a")
print("見つけた <a> の数:", len(all_links))

# STEP 2: ループで各リンクのテキストを取り出す(unit01 のリスト内包表記が使える)
texts = [a.get_text(strip=True) for a in all_links]
print("全リンクのテキスト:", texts)

# STEP 3: class="external" を含む <a> だけに絞り込む(class_ の末尾アンダースコアに注意)
external = soup.find_all("a", class_="external")
print("external なリンク数:", len(external))
print("そのテキスト       :", [a.get_text(strip=True) for a in external])

### 予測してみよう

次のセルは `NESTED`(座席案内のHTML)に対して `find_all("span")` と `find_all("div", class_="section")` を実行します。

**実行する前に予測**: `NESTED` の中に `<span>` はいくつあるでしょう?(section-name と seat-count が各セクションに1つずつ)。そして `class="section"` の `<div>` はいくつでしょう?

In [ ]:
# 予測してから実行!
soup = BeautifulSoup(NESTED, "html.parser")

spans = soup.find_all("span")
print("<span> の数        :", len(spans))
print("<span> のテキスト   :", [s.get_text(strip=True) for s in spans])

sections = soup.find_all("div", class_="section")
print('class="section" の数:', len(sections))

`find_all` が集めた件数と、ループ/内包表記での取り出しの組み合わせが、複数レコード抽出の土台です。

### 書いてみる

**課題**: `LINK_LIST` をパースした `soup` から、**全** `<a>` タグの**テキスト**を集め、その中で**文字数が3文字以下**のものだけをリストにして `result2` に入れてください。まず各リンクのテキストが何文字か数えて、どれが残るか予想してみましょう。

ヒント(概念レベル): `find_all("a")` で全リンクを取り、内包表記 `[a.get_text(strip=True) for a in ... if len(...) <= 3]` の形。テキストは "ホーム"(3文字)/"メニュー"(4文字)/"焙煎日記"(4文字)/"口コミサイト"(6文字)。3文字以下は1つだけ。

In [ ]:
soup = BeautifulSoup(LINK_LIST, "html.parser")

result2 = None
# ここに書く(result2 に代入する。find_all で全 <a> を取り、3文字以下のテキストだけ集める)


check("概念2: find_allと絞り込み", result2, ["ホーム"],
      hint='[a.get_text(strip=True) for a in soup.find_all("a") if len(a.get_text(strip=True)) <= 3]')

---
## 概念3: 属性アクセスとツリー移動(親子・兄弟)

### なぜ学ぶか
テキストだけでなく、**属性値**が主役になる場面は多いです:リンクの遷移先 `href`、要素を識別する `id`、画像の `src`。さらに「この見出しの**次に来る**要素」「この要素の**親**」のように、**ツリー上の位置関係**でたどりたいことも頻繁にあります。一覧→詳細ページのリンク収集(unit06)は、この属性アクセスとツリー移動の集大成です。

### 解説

**属性アクセス**(タグを辞書のように扱う):

- `tag["href"]` … `href` 属性の値を返す。C# の `Dictionary<string,string>` のインデクサ `dict["href"]` と同じ。**属性が無いと `KeyError`**(辞書と同じ)。
- `tag.get("href")` … 属性が無ければ `None` を返す安全版。C# の `TryGetValue` を1行にした感覚。unit01 で辞書の `get` を学んだのと同じ考え方。
- `tag.attrs` … 属性を全部辞書で返す(`{"class": [...], "href": "..."}`)。

**ツリー移動**(位置関係でたどる):

- `tag.parent` … 1つ上の親タグ。C# の `XElement.Parent`。
- `tag.find_next_sibling("div")` … **同じ親を持つ**次の兄弟のうち最初の `<div>` を返す(無ければ `None`)。「隣の要素」へ横移動するイメージ。
- `tag.find("span")` … `find`/`find_all` は soup 全体だけでなく**タグに対しても呼べる**。その場合「そのタグの**子孫**の中から」探します。「この section の中の span」のような**絞り込んでから中を探す**2段構えが定石です。

`find` で id 指定するには `soup.find("div", id="floor-1")` と書きます(class の `class_` と違い `id` はそのまま)。

In [ ]:
# GOAL: 属性を読み、ツリーを親子・兄弟方向にたどる

soup = BeautifulSoup(NESTED, "html.parser")

# STEP 1: 属性アクセス — floor-1 の div を id で探し、その id 属性を読む
floor1 = soup.find("div", id="floor-1")
print("floor1 の id 属性 :", floor1["id"])
print("get で安全に取得  :", floor1.get("id"), "/ 無い属性は:", floor1.get("href"))  # None

# STEP 2: タグの中をさらに探す(子孫検索)— floor-1 の中の最初の section-name
name_span = floor1.find("span", class_="section-name")
print("floor-1 内の最初のセクション名:", name_span.get_text(strip=True))

# STEP 3: 兄弟移動 — section-window の「次の兄弟 section」へ横移動
window = soup.find("div", id="section-window")
nxt = window.find_next_sibling("div")
print("section-window の次の兄弟の id:", nxt["id"])

# STEP 4: 親へ移動 — ある section の親 floor の id を得る
print("section-sofa の親 floor の id:", soup.find("div", id="section-sofa").parent["id"])

### 予測してみよう

次のセルは `section-counter`(1階の2番目のセクション)から `find_next_sibling("div")` で次の兄弟を探します。`section-counter` は floor-1 の**最後の** section です。

**実行する前に予測**: `section-counter` の次の兄弟 `<div>` は存在するでしょうか? しないなら返り値は何になるでしょう?(floor-2 は floor-1 の兄弟であって、section-counter の兄弟ではない点に注意)

In [ ]:
# 予測してから実行!
soup = BeautifulSoup(NESTED, "html.parser")

counter = soup.find("div", id="section-counter")
nxt = counter.find_next_sibling("div")
print("section-counter の次の兄弟 div:", repr(nxt))   # 兄弟がなければ None

`section-counter` は floor-1 内で最後なので次の兄弟 section は無く、`None` が返りました。「無ければ None」を前提にコードを書く癖が、壊れないスクレイパーの基本です。

### 書いてみる

**課題**: `BLOG_POST` をパースした `soup` から、`<a class="post-link">`(記事の詳細ページへのリンク)の **`href` 属性の値**を取り出して `result3` に入れてください(期待値: `"/blog/autumn-blend"`)。

ヒント(概念レベル): `soup.find("a")` で `<a>` タグを取り、`tag["href"]`(または `.get("href")`)で属性値を読む。このページの `<a>` は1つだけなので `find("a")` でOK。

In [ ]:
soup = BeautifulSoup(BLOG_POST, "html.parser")

result3 = None
# ここに書く(result3 に代入する。<a> タグの href 属性を取り出す)


check("概念3: 属性アクセス", result3, "/blog/autumn-blend",
      hint='soup.find("a")["href"] の形(または soup.find("a").get("href"))')

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| パースと find | `BeautifulSoup(html, "html.parser")` でツリー化、`find("h1")` で最初の1件、`.get_text(strip=True)` でテキスト | `XDocument.Parse` + `FirstOrDefault` |
| find_all と絞り込み | `find_all("a")` で全件リスト、`class_="external"` で絞り込み、ループ/内包表記で加工 | `Where(...).ToList()` |
| 属性とツリー移動 | `tag["href"]`/`.get()` で属性、`.parent`/`.find_next_sibling()`/子孫 `find` で位置移動 | `Dictionary` インデクサ / `XElement.Parent` |

**この先どこで使うか**:
- unit01 で「素の文字列処理は壊れやすい」と体感した課題を、今日 BeautifulSoup が解決しました。タグに属性が付いても順序が変わっても、`find`/`find_all` は構造で探すので壊れません。
- **unit03** では、今日の `find`/`find_all` を **CSSセレクタ**(`soup.select("div.floor .seat-count")`)に置き換えます。「floorクラスのdivの中のseat-count」のような**入れ子の絞り込みが1行で書ける**ようになり、今日の2段構え(絞り込んでから中を探す)がもっと楽になります。
- 属性アクセスとツリー移動は unit06 の「一覧ページからリンクを集めて詳細ページへ」で総動員されます。

**次**: 演習 `ex01_find_basic.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/web-scraping/unit02-beautifulsoup-basics/tests/test_ex01.py -q`